# SIFLOW · run_3 · MDLM eval + figures (fills Table 2)

Evaluates the SIFLOW student (k=1,2,4,8), the MDLM teacher step-curve, AR GPT-2, and optionally SDTT@8; builds the figures and auto-fills the LaTeX tables.

**Runtime:** comfortably under one Colab session.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

### Import the previous part(s)

This part needs the output zip(s) you downloaded earlier. Run the cell below — a file picker opens; select **all** of these at once:

- `run_1_data.zip` — val tokens for MAUVE + analyses
- `run_2_mdlm_ckpt.zip` — the trained MDLM head

*(If a long run here stopped early at the 11h guard, also re-upload **this** part's own checkpoint zip to resume.)* Using Drive instead? Set `USE_DRIVE=True` above and skip this.

In [ ]:
# === Import previous outputs (pick the .zip files listed above) ===
if not USE_DRIVE:
    from siflow.utils.io import import_zips
    import_zips(BASE)
else:
    print('USE_DRIVE: reading prior outputs from', BASE)

In [ ]:
!python scripts/evaluate.py --config siflow/config/mdlm.yaml --system siflow \
    --ckpt-dir {BASE}/ckpt/mdlm --ref-tokens {BASE}/data/owt_gpt2_val.npy \
    --n-samples 1000 --out {BASE}/results/mdlm.json

In [ ]:
!python scripts/evaluate.py --config siflow/config/mdlm.yaml --system teacher \
    --steps 8 32 64 1024 --ref-tokens {BASE}/data/owt_gpt2_val.npy \
    --n-samples 1000 --out {BASE}/results/mdlm_teacher.json

In [ ]:
!python scripts/evaluate.py --config siflow/config/mdlm.yaml --system ar --ar-model gpt2 \
    --ref-tokens {BASE}/data/owt_gpt2_val.npy --n-samples 1000 --out {BASE}/results/ar_gpt2.json

In [ ]:
# (optional) SDTT@8 baseline -- skips gracefully if unavailable
try:
    !pip -q install git+https://github.com/jdeschena/sdtt.git
    import torch, json
    from sdtt import load_small_student
    from siflow.eval.gen_ppl import GPT2Scorer, decode_ids
    from siflow.eval.diversity import diversity_metrics
    from transformers import AutoTokenizer
    m = load_small_student(loss="kld", round=7).cuda().eval()
    tok = AutoTokenizer.from_pretrained("gpt2"); texts = []
    while len(texts) < 1000:
        s = m.sample(n_samples=64, num_steps=8, seq_len=256)
        texts += decode_ids(s if torch.is_tensor(s) else torch.tensor(s), tok)
    sc = GPT2Scorer("gpt2-large")
    json.dump({"run_id":"sdtt","method":"SDTT","source":"reproduced",
               "metrics":{"steps=8":{**sc.perplexity(texts),**diversity_metrics(texts),"nfe":8}}},
              open(f"{BASE}/results/sdtt.json","w"), indent=2)
    print("SDTT done")
except Exception as e:
    print("SDTT skipped:", e)

In [ ]:
!python scripts/make_figures.py --results {BASE}/results --train-log {BASE}/ckpt/mdlm/train_log.jsonl --out-dir {BASE}/figures
!python scripts/make_tables.py --results {BASE}/results --out {BASE}/tables_auto.tex
print(open(f"{BASE}/tables_auto.tex").read()[:1500])

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'run_3_results.zip', include=['results', 'figures', 'tables_auto.tex'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)